In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

In [ ]:
# Load preprocessed data
df = pd.read_csv("cleaned_data_preprocessing.csv")

In [ ]:
# Split dataset into before and after mandate periods
df_before = df[df['within_mandate_period'] == 0].copy()
df_after  = df[df['within_mandate_period'] == 1].copy()

In [ ]:
# Define feature sets for different prediction tasks
# Mask-wearing prediction: remove outcome variables and identifiers
mask_drop_cols = [
    'RecordNo',
    'endtime',
    'face_mask_behaviour_scale',
    'face_mask_behaviour_binary',
    'protective_behaviour_scale',
    'protective_behaviour_binary'
]

X_mask_before = df_before.drop(columns=mask_drop_cols)
y_mask_before = df_before['face_mask_behaviour_binary']

X_mask_after = df_after.drop(columns=mask_drop_cols)
y_mask_after = df_after['face_mask_behaviour_binary']

In [ ]:
# General protective behaviour prediction
general_drop_cols = [
    'RecordNo',
    'endtime',
    'face_mask_behaviour_scale',
    'face_mask_behaviour_binary',
    'protective_behaviour_scale',
    'protective_behaviour_binary',
    'protective_behaviour_nomask_scale'
]

X_general_before = df_before.drop(columns=general_drop_cols)
y_general_before = df_before['protective_behaviour_binary']

X_general_after = df_after.drop(columns=general_drop_cols)
y_general_after = df_after['protective_behaviour_binary']

In [ ]:
# Train-test split
# Maintain class distribution in train/test sets
X_train_mb, X_test_mb, y_train_mb, y_test_mb = train_test_split(
    X_mask_before, y_mask_before,
    test_size=0.2,
    random_state=10,
    stratify=y_mask_before
)

X_train_ma, X_test_ma, y_train_ma, y_test_ma = train_test_split(
    X_mask_after, y_mask_after,
    test_size=0.2,
    random_state=10,
    stratify=y_mask_after
)

X_train_gb, X_test_gb, y_train_gb, y_test_gb = train_test_split(
    X_general_before, y_general_before,
    test_size=0.2,
    random_state=10,
    stratify=y_general_before
)

X_train_ga, X_test_ga, y_train_ga, y_test_ga = train_test_split(
    X_general_after, y_general_after,
    test_size=0.2,
    random_state=10,
    stratify=y_general_after
)


In [ ]:
# Convert target variable from Yes/No to binary (1/0)
def encode_target(y):
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]
    return y.astype(str).str.strip().map({"No": 0, "Yes": 1})

In [ ]:
# Save datasets in a consistent format for modelling
def save_dataset_flat(base_path, prefix, X_train, y_train, X_test, y_test):
    os.makedirs(base_path, exist_ok=True)

    y_train_enc = encode_target(y_train)
    y_test_enc = encode_target(y_test)

    X_train.to_csv(f"{base_path}/{prefix}_X_train.csv", index=False)
    y_train_enc.to_csv(f"{base_path}/{prefix}_y_train.csv", index=False)

    X_test.to_csv(f"{base_path}/{prefix}_X_test.csv", index=False)
    y_test_enc.to_csv(f"{base_path}/{prefix}_y_test.csv", index=False)


In [ ]:
# Save datasets for all tasks
base_path = "data"

save_dataset_flat(base_path, "mask_before", X_train_mb, y_train_mb, X_test_mb, y_test_mb)
save_dataset_flat(base_path, "mask_after", X_train_ma, y_train_ma, X_test_ma, y_test_ma)
save_dataset_flat(base_path, "general_before", X_train_gb, y_train_gb, X_test_gb, y_test_gb)
save_dataset_flat(base_path, "general_after", X_train_ga, y_train_ga, X_test_ga, y_test_ga)